In [1]:
import MetricsReloaded as MR
import nibabel as nib
import pandas as pd
import numpy as np

# Comparison of two segmentations

In this part of the notebook, we will get started with a simple comparison of two segmentations, using as metrics the dice score and the average symmetric surface distance. 
We will proceed in two step
- Loading of the data
- Comparison of the resulting segmentation maps

In [2]:
#Step 1

ref = nib.load('reference/reference_1.nii.gz')
ref_data = ref.get_fdata()
pred = nib.load('prediction/prediction_1.nii.gz')
pred_data = pred.get_fdata()

# The prediction data is binarised to be used for the binary pairwise measurements
pred_data_bin = np.where(pred_data>=0.5, np.ones_like(pred_data),np.zeros_like(pred_data))

print(np.sum(ref_data), np.sum(pred_data), ref.header.get_zooms(), pred.header.get_zooms(), ref_data.shape, pred_data.shape)

881.0 677.9561501774587 (1.0, 1.0, 1.0) (1.0, 1.0, 1.0) (24, 256, 256) (24, 256, 256)


In [3]:
np.unique(pred_data)

array([1.66901392e-24, 6.34210139e-22, 7.44894266e-22, ...,
       9.99999762e-01, 9.99999881e-01, 1.00000000e+00])

In [4]:
from MetricsReloaded.metrics.pairwise_measures import BinaryPairwiseMeasures as BPM

bpm = BPM(pred_data_bin, ref_data, measures=['dsc', 'assd'], pixdim=ref.header.get_zooms())

There are different ways of accessing the results. If you know the individual functions' names, you can do:

In [5]:
print(bpm.dsc(), bpm.measured_average_distance())

0.6812339331619537 0.8546716488049553


/Users/carolesudre/Development/MetricsReloaded/MetricsReloaded/metrics/pairwise_measures.py:1341: UserWarning: Percentile not specified in options for Hausdorff distance - default set to 95
  warnings.warn('Percentile not specified in options for Hausdorff distance - default set to 95')


Alternatively, the results can also be extracted from the result dictionary

In [6]:
bpm.to_dict_meas()

{'dsc': 0.6812339331619537, 'assd': 0.8546716488049553}

# Comparison of two sets of segmentations - all cases with existing prediction output
We use the matching_cases.csv file to identify the pairs of reference and prediction

## Reading and preparation of the data
- Read and load all the reference data
- Read, load and binarise all the prediction data
- Prepare the metrics assessment using again Dice and ASSD

In [7]:
df_data = pd.read_csv('matching_cases.csv')
list_ref = [nib.load(k).get_fdata() for k in list(df_data['reference'])]
list_pred = [nib.load(k).get_fdata() for k in list(df_data['prediction'])]
list_pred_bin = [np.where(k>=0.5,np.ones_like(k),np.zeros_like(k)) for k in list_pred]



To get directly the list of metrics values, one can use simply the MuliLabelPairwiseMeasures

In [8]:
from MetricsReloaded.processes.mixed_measures_processes import MultiLabelPairwiseMeasures as MLP

mlp = MLP(list_pred_bin, list_ref, list_pred, list_values=[1], names=list(df_data['prediction']),
          measures_overlap=['dsc'], measures_boundary=['assd'],per_case=True)

In [9]:
df_bin, _ = mlp.per_label_dict()
df_bin

1  is treated label


/Users/carolesudre/Development/MetricsReloaded/MetricsReloaded/metrics/pairwise_measures.py:1341: UserWarning: Percentile not specified in options for Hausdorff distance - default set to 95
  warnings.warn('Percentile not specified in options for Hausdorff distance - default set to 95')


,dsc,assd,label,case,worse_dist,check_empty
0,0.710943,0.568781,1,prediction/prediction_0.nii.gz,362.833295,None
1,0.681234,0.854672,1,prediction/prediction_1.nii.gz,362.833295,None
2,0.643046,1.170390,1,prediction/prediction_2.nii.gz,362.833295,None
3,0.855279,0.306300,1,prediction/prediction_3.nii.gz,362.833295,None
4,0.874437,0.255695,1,prediction/prediction_4.nii.gz,362.833295,None
5,0.852987,0.239211,1,prediction/prediction_5.nii.gz,362.833295,None
6,0.811450,0.435307,1,prediction/prediction_6.nii.gz,362.833295,None
7,0.873595,0.265076,1,prediction/prediction_7.nii.gz,362.833295,None
8,0.867885,0.337863,1,prediction/prediction_8.nii.gz,362.833295,None
9,0.819682,0.427100,1,prediction/prediction_9.nii.gz,362.833295,None


Alternatively, one can also use the full process pathway to get to a final dataframe with the results. This requires to pass in a dictionary containing the relevant information for the comparison process. For a semantic segmentation problem, the following fields are required:

- pred_class
- ref_class
- pred_proba
- list_values
- name

In [10]:
data_dict = {'pred_class': list_pred_bin,
            'ref_class': list_ref,
            'pred_prob': list_pred,
            'list_values': [1],
            'names':list(df_data['reference'])
            }


In [11]:
from MetricsReloaded.processes.overall_process import ProcessEvaluation as PE

pe = PE(data_dict, category='SemS', measures_overlap=['dsc'],measures_boundary=['assd'])
pe.resseg

1  is treated label


/Users/carolesudre/Development/MetricsReloaded/MetricsReloaded/metrics/pairwise_measures.py:1341: UserWarning: Percentile not specified in options for Hausdorff distance - default set to 95
  warnings.warn('Percentile not specified in options for Hausdorff distance - default set to 95')


,dsc,assd,label,case,worse_dist,check_empty,assd_nanrep,dsc_nanrep
0,0.710943,0.568781,1,reference/reference_0.nii.gz,362.833295,None,0.568781,0.710943
1,0.681234,0.854672,1,reference/reference_1.nii.gz,362.833295,None,0.854672,0.681234
2,0.643046,1.170390,1,reference/reference_2.nii.gz,362.833295,None,1.170390,0.643046
3,0.855279,0.306300,1,reference/reference_3.nii.gz,362.833295,None,0.306300,0.855279
4,0.874437,0.255695,1,reference/reference_4.nii.gz,362.833295,None,0.255695,0.874437
5,0.852987,0.239211,1,reference/reference_5.nii.gz,362.833295,None,0.239211,0.852987
6,0.811450,0.435307,1,reference/reference_6.nii.gz,362.833295,None,0.435307,0.811450
7,0.873595,0.265076,1,reference/reference_7.nii.gz,362.833295,None,0.265076,0.873595
8,0.867885,0.337863,1,reference/reference_8.nii.gz,362.833295,None,0.337863,0.867885
9,0.819682,0.427100,1,reference/reference_9.nii.gz,362.833295,None,0.427100,0.819682


In [12]:
# Comparison of two sets of segmentation when some cases are not predicted 

In [13]:
df_data_nonmis = df_data.dropna()
list_mis = [k for k in list(df_data['reference']) if k not in list(df_data_nonmis['reference'])]

In [14]:
list_ref_nonmis = [nib.load(k).get_fdata() for k in list(df_data_nonmis['reference'])]
list_pred_nonmis = [nib.load(k).get_fdata() for k in list(df_data_nonmis['prediction'])]
list_pred_bin_nonmis = [np.where(k>=0.5,np.ones_like(k),np.zeros_like(k)) for k in list_pred]
list_ref_mis = [nib.load(k).get_fdata() for k in list_mis]

In [15]:
data_dict_mis = {'pred_class': list_pred_bin_nonmis,
            'ref_class': list_ref_nonmis,
            'pred_prob': list_pred_nonmis,
            'list_values': [1],
            'names':list(df_data_nonmis['reference']),
             'ref_missing_pred': list_ref_mis,
                 'missing_names': list_mis
            }

In [16]:
pe_mis = PE(data_dict_mis, category='SemS', measures_overlap=['dsc'],measures_boundary=['assd'])
pe_mis.resseg

1  is treated label


/Users/carolesudre/Development/MetricsReloaded/MetricsReloaded/metrics/pairwise_measures.py:1341: UserWarning: Percentile not specified in options for Hausdorff distance - default set to 95
  warnings.warn('Percentile not specified in options for Hausdorff distance - default set to 95')


[]
[]
[]
[]
[]
Nothing in first
Performing concatenation
Nothing in first
Nothing in first
label  not present


,dsc,assd,label,case,worse_dist,check_empty,assd_nanrep,dsc_nanrep
0,0.710943,0.568781,1,reference/reference_0.nii.gz,362.833295,None,0.568781,0.710943
1,0.023681,8.930785,1,reference/reference_2.nii.gz,362.833295,None,8.930785,0.023681
2,0.023351,6.855434,1,reference/reference_3.nii.gz,362.833295,None,6.855434,0.023351
3,0.082344,4.343974,1,reference/reference_4.nii.gz,362.833295,None,4.343974,0.082344
4,0.071065,5.849462,1,reference/reference_5.nii.gz,362.833295,None,5.849462,0.071065
5,0.003997,8.947237,1,reference/reference_6.nii.gz,362.833295,None,8.947237,0.003997
6,0.018266,6.880306,1,reference/reference_7.nii.gz,362.833295,None,6.880306,0.018266
7,0.053213,3.559748,1,reference/reference_8.nii.gz,362.833295,None,3.559748,0.053213
8,0.005692,9.117238,1,reference/reference_10.nii.gz,362.833295,None,9.117238,0.005692
9,0.026679,5.956536,1,reference/reference_12.nii.gz,362.833295,None,5.956536,0.026679
